# Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append("/home/gliu2/contrastive_rl")

import copy
import functools

from matplotlib import pyplot as plt
from matplotlib import animation
from mpl_toolkits.axes_grid1 import make_axes_locatable
from IPython.display import HTML

import jax
import optax
import numpy as np
from acme import specs
import tensorflow as tf

from acme.tf.savers import SaveableAdapter

from contrastive.config import ContrastiveConfig
from contrastive import utils as contrastive_utils
from contrastive import make_networks
from contrastive.utils import make_environment
from contrastive import ContrastiveLearner

# disable tensorflow_probability warning: The use of `check_types` is deprecated and does not have any effect.
import logging
logger = logging.getLogger("root")

class CheckTypesFilter(logging.Filter):
    def filter(self, record):
        return "check_types" not in record.getMessage()

logger.addFilter(CheckTypesFilter())

from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
import seaborn as sn
import pandas as pd

import pickle
# CHANGE ME
from tqdm import tqdm

## Overlaying visitation and psi norms

In [ ]:
# (don't modify)
from utils_exploration_psi_norms import eval_and_get_cells_visited, get_psi_norms
NUM_AXES = 3
NUM_EPISODES = 5
EPISODE_LENGTH = 150
ckpt_list = [1,2,3,4,5,10,15,20,30]
log_dir = '/home/gliu2/rebuttal/cpc/'
env_name = 'sawyer_bin'
goal_type = 'fixed'
seed = 3

# psi norms

In [ ]:
ckpt_list = [1,2,3,4]
for ckpt_num in tqdm(ckpt_list):
    print(f"checkpoint {ckpt_num}")
    print("-----------------------------------------------------------")
    axes_names = ['x', 'y', 'z']
    if env_name == 'sawyer_bin':
        axes_lims = {
            0: (-0.3, 0.25),
            1: (0.6, 0.9),
            2: (0.01, 0.15),
        }
    else:
        assert env_name == 'sawyer_box'
        axes_lims = {
            0: (-0.3, 0.25),
            1: (0.45, 0.9),
            2: (0.01, 0.16),
        }

    # roll out + compute sample fixed goal trajectories
    returns_dict, success_rates_dict, positions_dict, goals_dict, images_dict, cells = eval_and_get_cells_visited(env_name, log_dir, goal_type, seed, ckpt_num, grid_width=0.01, render_images=True, NUM_EPISODES=NUM_EPISODES)

    # compute psi norms
    goal_locations, psi_norms, critic = get_psi_norms(env_name, log_dir, goal_type, seed, ckpt_num)

    # optional: save data
    # fname = 'data_repr/data_exploration_video_{}_{}_seed_{}_ckpt_{}.pickle'.format(env_name, goal_type, seed, ckpt_num)
    # with open(fname, 'wb') as f:
    #     pickle.dump([goal_locations, psi_norms, critic, returns_dict, success_rates_dict, positions_dict, goals_dict, images_dict, cells], f)
    # print('Saved data to', fname)

    pos_arr = np.array(positions_dict[seed]).reshape(-1, NUM_AXES)
    pos_id_arr = np.broadcast_to(np.expand_dims(np.arange(NUM_EPISODES), 1), (NUM_EPISODES, EPISODE_LENGTH)).flatten()
    goals_arr = np.array(goals_dict[seed])
    starts_arr = np.array(positions_dict[seed])[:, 0, :]

    for idx1, idx2 in [(0, 1), (0, 2), (1,2)]:
        fig, ax = plt.subplots()
        psi_norms_heatmap = plt.scatter(x=goal_locations[:, 4+idx1], y=goal_locations[:, 4+idx2], c=np.log(psi_norms), s=8, alpha=0.2, cmap='coolwarm')
        for epi in range(NUM_EPISODES):
            epi_slice = slice(epi*150, (epi+1)*150)
            plt.plot(pos_arr[epi_slice, idx1], pos_arr[epi_slice, idx2], label=epi)#, c=[epi]*150, cmap='viridis'
            if success_rates_dict[seed][epi]:
                plt.annotate('✔️', (pos_arr[(epi+1)*150-1, idx1], pos_arr[(epi+1)*150-1, idx2]), c='green')
        plt.scatter(starts_arr[:, idx1], starts_arr[:, idx2], c='red', marker='D', s=30)
        plt.scatter(goals_arr[:, idx1], goals_arr[:, idx2], c='orange', marker='*', s=80)
        plt.xlabel(axes_names[idx1])
        plt.ylabel(axes_names[idx2])
        plt.xlim(axes_lims[idx1])
        plt.ylim(axes_lims[idx2])
        legend = ax.legend(*psi_norms_heatmap.legend_elements(), loc='lower right', title='log psi norms')
        ax.add_artist(legend)
        # plt.legend(title='episode', loc='lower right')
        plt.show()
    low_z_threshold = 0.002

    for low_z in [0.04, 0.08, 0.12, 0.16]:#[0.02, 0.03, 0.04, 0.05, 0.06, 0.08, 0.10, 0.12, 0.14]:
        # compute psi norms at low z
        axes_lims_low = axes_lims
        axes_lims_low[2] = (low_z - low_z_threshold, low_z + low_z_threshold)
        goal_locations_low, psi_norms_low, critic_low = get_psi_norms(env_name, log_dir, goal_type, seed, ckpt_num, axes_lims=axes_lims_low)
        pos_arr
        
        
        # plot
        # requires precomputed (pos_arr, starts_arr, goals_arr), (goal_locations, psi_norms)
        for idx1, idx2 in [(0, 1)]:
            fig, ax = plt.subplots()
            # sawyer bin vmin / vmax
            # 3.2 - 5 for ckpt 20 and RG ckpt 10
            # 5 - 7.5 for FG ckpt 10
            
            # sawyer box vmin / bmax
            # 3 - 4 for ckpt 10
            
            psi_norms_heatmap = plt.scatter(x=goal_locations_low[:, 4+idx1], y=goal_locations_low[:, 4+idx2], c=np.log(psi_norms), s=8, alpha=0.2, cmap='coolwarm', vmin=4.5, vmax=6.5)
            for epi in range(NUM_EPISODES):
                epi_slice = slice(epi*150, (epi+1)*150)
                plt.plot(pos_arr[epi_slice, idx1], pos_arr[epi_slice, idx2], label=epi)#, c=[epi]*150, cmap='viridis'
                if success_rates_dict[seed][epi]:
                    plt.annotate('✔️', (pos_arr[(epi+1)*150-1, idx1], pos_arr[(epi+1)*150-1, idx2]), c='green')
            plt.scatter(starts_arr[:, idx1], starts_arr[:, idx2], c='red', marker='D', s=30)
            plt.scatter(goals_arr[:, idx1], goals_arr[:, idx2], c='orange', marker='*', s=80)
            plt.xlabel(axes_names[idx1])
            plt.ylabel(axes_names[idx2])
            plt.xlim(axes_lims[idx1])
            plt.ylim(axes_lims[idx2])
            legend = ax.legend(*psi_norms_heatmap.legend_elements(), loc='upper left', title='log psi norms')
            ax.add_artist(legend)
            plt.legend(title='episode', loc='lower right')
            plt.title('z = {}'.format(low_z))
            plt.show()